# Figure 3 — what transfers through π

π appears to carry areal position on the cortical hierarchy, along with the properties that vary
across that axis. Properties that vary through the cortical depth do not come across.

Nothing here requires re-fitting. Each result routes a mouse map through the frozen coupling and
correlates it with an independent human map, so the notebook recomputes them.

The null matters more than the correlation. Two smooth brain maps correlate with each other
readily, so the question is whether it is this coupling that carries the signal. Every test below
uses `translation_spin_null`, which spins the mouse map and routes it through the real π. That
preserves both spatial autocorrelation and the coupling, and breaks only the specific mouse→human
correspondence. Nulls that shuffle region labels destroy spatial autocorrelation and will report
relationships that do not survive a spin.

Before running: `python scripts/fetch_data.py`. Logs are verified rather than trusted.

In [ ]:
import importlib.util, json, subprocess, sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
FIGS = ROOT.parent / 'manuscript' / 'figures'
sys.path.insert(0, str(ROOT / 'src'))

from otter.data import load_pi, pi_provenance
from otter.eval.nulls import translation_spin_null

pi = load_pi()
PROV = pi_provenance()
LOGS = ROOT / 'outputs' / 'logs'
print(f"coupling {pi.shape[0]:,} x {pi.shape[1]:,}   {PROV['pi_file']}   sha {PROV['pi_sha256'][:16]}...")

PUBLISHED = {
    'myelin from mouse T1w:T2w (r)':      (0.47, 0.02),
    'myelin from mouse cytoarch (r)':     (0.47, 0.02),
    'routed Schaefer regions':            (388,  1),
    'networks top-matching homologue':    (6,    0),
    'network spin p':                     (0.002, 0.002),
    'marker mean r':                      (0.23, 0.02),
    'contrast mean r':                    (0.07, 0.02),
}

def check(name, value):
    exp, tol = PUBLISHED[name]
    ok = abs(value - exp) <= tol
    print(f"  [{'ok ' if ok else 'FAIL'}] {name:34s} computed {value:.4g}   manuscript {exp}")
    return ok


def verified_log(fname):
    '''Read a log only after confirming which coupling produced it.'''
    d = json.loads((LOGS / fname).read_text())
    sha = d.get('pi_sha256') if isinstance(d, dict) else None
    if sha is None:
        print(f"  {fname}: NO coupling provenance recorded")
    elif sha != PROV['pi_sha256']:
        raise RuntimeError(f"{fname} was built on a DIFFERENT coupling ({sha[:16]}...)")
    else:
        print(f"  {fname}: provenance verified")
    return d


def load_experiment(relpath):
    '''Import an experiment module by path (their filenames start with digits).'''
    p = ROOT / 'experiments' / relpath
    spec = importlib.util.spec_from_file_location(p.stem.lstrip('0123456789_'), p)
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    return m

## 1. Does microstructure translate? (Fig. 3c)

Two independent mouse measurements, a T1w:T2w myelin proxy and cytoarchitectural type, are routed
through π and compared against the human HCP myelin map, which the model never saw.

This is the result that reversed when the coupling became canonical and a bug in the gradient
component was fixed. An earlier draft of §3 argued that translation was connectional and not
microstructural. It is both. The values are recomputed here from the raw maps rather than quoted,
since this is the claim in the paper most worth being able to re-derive.

In [ ]:
F = load_experiment('fulcher_2019_multimodal_gradient/01_gradient_validation.py')

parcel_acr = F.load_mouse_parcel_acronyms()
node_region = F.load_human_node_region()
myelin_reg = F.load_human_myelin()

def route_modality(value_by_acr):
    vec = np.array([value_by_acr.get(a, np.nan) for a in parcel_acr])
    mask = np.isfinite(vec)
    return vec, mask, F.aggregate_to_regions(F.route_through_pi(vec, pi, mask), node_region)

t1t2_vec, t1t2_mask, pred_t1t2 = route_modality(F.load_mouse_t1t2())
cyto_vec, cyto_mask, pred_cyto = route_modality(F.load_mouse_cytoarch())

r_t1t2 = F._corr(pred_t1t2, myelin_reg)
r_cyto = F._corr(pred_cyto, myelin_reg)
n_regions = int(np.isfinite(pred_t1t2).sum())

print(f"routed territory: {n_regions} of 400 Schaefer regions\n")
print(f"mouse T1w:T2w        -> human myelin   r = {r_t1t2[0]:+.3f}  (n = {r_t1t2[3]})")
print(f"mouse cytoarchitecture -> human myelin r = {r_cyto[0]:+.3f}  (n = {r_cyto[3]})\n")
check('myelin from mouse T1w:T2w (r)', abs(r_t1t2[0]))
check('myelin from mouse cytoarch (r)', abs(r_cyto[0]))
check('routed Schaefer regions', n_regions)

## 2. The transfer battery (Fig. 3d, 3e)

The battery groups fourteen properties by their relation to the areal hierarchy: the macroscale
hierarchy maps themselves, properties that vary along that hierarchy, and properties orthogonal
to it (laminar contrasts, spatially uniform cell classes). The claim is a dissociation, so it
needs the negatives as much as the positives.

Each entry is a routed translation with a spin null, so the cost is modest. `RUN_EXPERIMENTS =
True` recomputes the battery from the raw data in tens of minutes; otherwise the logs are
provenance-checked.

In [ ]:
RUN_EXPERIMENTS = False

BATTERY = {
    'published_map_validation.json':      'validation/00_validate_published_maps.py',
    'biccn_contrast_reframe.json':        'biccn_2023_cell_types/03_contrast_reframe.py',
    'hodge_areal_type_reframe.json':      'hodge_2019_cortical_layers/03_areal_type_reframe.py',
    'margulies_2016_gradient.json':       None,      # written by the TransBrain benchmark
    'fulcher_2019_gradient.json':         None,      # written by the section-5 coverage nulls
}

if RUN_EXPERIMENTS:
    for log, rel in BATTERY.items():
        if rel is None:
            print(f'{log}: produced elsewhere, skipping'); continue
        print(f're-running {rel} ...')
        r = subprocess.run([sys.executable, str(ROOT / 'experiments' / rel)], cwd=str(ROOT),
                           capture_output=True, text=True)
        print((r.stdout or r.stderr)[-300:])

for log in BATTERY:
    verified_log(log)

In [ ]:
# Panel e recomputed from the logs: individual layer-marker genes (areal signal retained) versus
# layer contrasts built from the SAME genes (areal signal removed). The contrast is the control --
# if translation carried laminar rather than areal information, contrasts would survive.
# Granular L4 - infragranular is the expected exception: cortical granularity IS the areal
# hierarchy, so that contrast does not remove the areal signal.
#
# Parsed exactly as make_fig3_marker_contrast.py does. The three sources are separate logs, and
# the contrast entries sit at the TOP level of hodge_areal_type_reframe.json rather than under a
# 'contrasts' key -- guessing the schema here silently produced no output.
mk = verified_log('hodge_2019_layer_markers.json')
areal = verified_log('hodge_areal_type_reframe.json')
refined = verified_log('hodge_2019_layer_markers_refined.json')

markers = [(m['gene'], float(m['pearson_r']), float(m['empirical_p'])) for m in mk['markers']]
contrasts = []
for key, lab in [('granular_L4_minus_infragranular', 'L4 - infragranular'),
                 ('supragranular_minus_infragranular', 'supra - infragranular'),
                 ('supragranular_minus_granular', 'supra - granular')]:
    d = areal[key]
    contrasts.append((lab, float(d.get('pearson_r', d.get('r'))), float(d['spin_p'])))
ud = refined.get('upper_minus_deep_contrast')
if ud:
    contrasts.append(('upper - deep', float(ud.get('pearson_r', ud.get('r'))),
                      float(ud.get('empirical_p', ud.get('spin_p')))))

mr = np.array([r for _, r, _ in markers]); msig = sum(1 for _, _, q in markers if q < 0.05)
cr = np.array([r for _, r, _ in contrasts]); csig = sum(1 for _, _, q in contrasts if q < 0.05)
print(f"\nlayer markers   n={len(mr)}  mean r {mr.mean():+.3f}  significant {msig}/{len(mr)}")
print(f"layer contrasts n={len(cr)}  mean r {cr.mean():+.3f}  significant {csig}/{len(cr)}\n")
for lab, r, q in contrasts:
    print(f"   {lab:24s} r={r:+.3f}  p={q:.3f}{'   <- granularity IS the hierarchy' if 'L4' in lab else ''}")
print()
check('marker mean r', float(mr.mean()))
check('contrast mean r', float(cr.mean()))

## 3. Networks (Fig. 3a, 3b)

Routing the mouse resting-state networks of Coletta et al. through π assigns 6 of 10 to their
like-named human network, against a spin-null expectation of 1.0.

In [ ]:
col = verified_log('coletta_2020_cross_species_rsn.json')
fair = verified_log('fair_nulls_coletta_test2c.json')['coletta']
sub = col['sub_test_A_labeled_correspondence']

print(f"\n{sub['n_diagonal_argmax']}/{sub['n_pairs_scored']} mouse networks top-match their homologue")
print(f"spin null mean {fair['spin_null_mean']:.2f}   p = {fair['spin_p']:.3f}\n")
check('networks top-matching homologue', sub['n_diagonal_argmax'])
check('network spin p', fair['spin_p'])

for p in sub['per_pair_scores']:
    mark = 'on homologue' if p['is_argmax_diagonal'] else f"drifts -> {p['argmax_human_net']}"
    print(f"   {p['mouse_net']:14s} {mark}")

## 4. Build the figure panels

One script per published panel, in `manuscript/figures/fig3/`. Filenames carry the panel letter
they appear as; `3b_network_brains` was previously called `3a_network_brains` while sitting in
panel b.

In [ ]:
RUN_PANELS = True

PANELS = [('a', 'make_fig3a_network_matrix.py'),
          ('b', 'make_fig3_network_brains.py'),
          ('c', 'make_fig3_myelin_brains.py'),
          ('d', 'make_fig3_transfer_battery.py'),
          ('e', 'make_fig3_marker_contrast.py')]

if RUN_PANELS:
    for panel, script in PANELS:
        r = subprocess.run([sys.executable, str(FIGS / 'fig3' / script)], cwd=str(ROOT),
                           capture_output=True, text=True)
        print(f"panel {panel}  {script:34s} {'ok' if r.returncode == 0 else 'FAILED'}")
        if r.returncode != 0:
            print((r.stderr or '')[-400:])
else:
    print('skipped; set RUN_PANELS = True')

## 5. Where §3 stands

Translation follows the areal hierarchy. Properties that vary along that axis come across: the
hierarchy maps themselves, cell-class densities that track it, and the FC gradient. Properties
orthogonal to it do not, with laminar contrasts and spatially uniform cell classes all failing
their spin nulls. The dissociation is the result rather than any single correlation.

In [ ]:
print(f"coupling              {PROV['pi_file']}")
print(f"myelin translation    r = {abs(r_t1t2[0]):.2f} (T1w:T2w), {abs(r_cyto[0]):.2f} (cytoarchitecture)")
print(f"routed territory      {n_regions} of 400 Schaefer regions")
print(f"network correspondence {sub['n_diagonal_argmax']}/{sub['n_pairs_scored']}, spin p = {fair['spin_p']:.3f}")
print()
ok = all([check('myelin from mouse T1w:T2w (r)', abs(r_t1t2[0])),
          check('myelin from mouse cytoarch (r)', abs(r_cyto[0])),
          check('routed Schaefer regions', n_regions),
          check('networks top-matching homologue', sub['n_diagonal_argmax']),
          check('network spin p', fair['spin_p'])])
print('\nALL CHECKS PASS' if ok else '\nSOME CHECKS FAILED -- text and code have diverged')